# 02 — Canonical tables: cleaned population, trace cache, CV splits

**What this notebook does.** Three deliverables, in order:

1. **Clean the working population.** Drop the 15 L1 neurons (`WORKFLOW §2.1` only
   covers L2/3–L6) and add the two label columns the project will predict:
   `layer_label` for Phase 1 (geometric layer from the structural segmenter)
   and `celltype_label` for Phase 3 (the AIBS metamodel cell-type vocabulary).

2. **Cache calcium traces and trial-level behaviour from the H5 once.** This is
   the substrate for every later feature tier (Tier A amplitude / shape, Tier B
   reliability, Tier E temporal traces for Phase 2). One pass over the H5,
   session by session, with per-session checkpoint files so the read can be
   resumed if interrupted. From this we also build the within-hash trial-
   averaged trace per `(nucleus_id, condition_hash)` — the canonical estimator
   of the stimulus-locked response per `WORKFLOW §3.3`.

3. **Define the CV splits.** `StratifiedGroupKFold(5)` grouped by `nucleus_id`
   and stratified by `layer_label`, plus `LeaveOneSessionOut` over the 13
   active session-scan blocks. Saved as a single fold-assignment parquet that
   every later modelling notebook reads.

**Reads.**
- `data/processed/tables/units_v1_exc_best.parquet` (from notebook 01)
- `data/functional/microns_functional.h5`

**Writes.**
- `data/processed/tables/units_working.parquet`
- `data/processed/tables/trials_meta.parquet`
- `data/processed/tables/traces.parquet`
- `data/processed/tables/traces_avg.parquet`
- `data/processed/splits/cv_assignments.parquet`
- per-session checkpoints under `data/interim/traces_per_session/`

**Methodological notes.**
- The trace cache stores **calcium responses per `(neuron, trial)`** and
  **shared trial-level behaviour per `(session, trial)`** in two separate
  parquets, so behaviour is not duplicated across the 8.9k neurons of a trial.
- Within-hash averaging is the only averaging this notebook does. Across-hash
  averaging is forbidden by `WORKFLOW §3.3` and never appears here.
- Trial-level behavioural summaries (Tier C0) and behaviour-conditioned
  features (Tier C1) are NOT computed here — they come in a later notebook.
  This notebook only caches the raw traces those features will read.

## 1. Setup

In [1]:
from __future__ import annotations

import sys, os, json, time, gc
from pathlib import Path
from collections import Counter, defaultdict

if '..' not in sys.path:
    sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import h5py
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

import microns_datacleaner as mic

from src.config import (
    DATA_DIR, PROCESSED_DIR, INTERIM_DIR,
    PROCESSED_TABLES_DIR, PROCESSED_SPLITS_DIR,
    FUNCTIONAL_H5,
    RANDOM_SEED, MICRONS_VERSION,
    STIMULUS_FAMILIES,
    ensure_dirs,
)

ensure_dirs()
np.random.seed(RANDOM_SEED)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

INTERIM_TRACES_DIR = INTERIM_DIR / 'traces_per_session'
INTERIM_TRACES_DIR.mkdir(parents=True, exist_ok=True)

print('cwd          :', Path.cwd())
print('DATA_DIR     :', DATA_DIR)
print('FUNCTIONAL_H5:', FUNCTIONAL_H5, '| exists?', FUNCTIONAL_H5.exists())
print('checkpoint dir:', INTERIM_TRACES_DIR)

cwd          : /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project
DATA_DIR     : data
FUNCTIONAL_H5: data/functional/microns_functional.h5 | exists? True
checkpoint dir: data/interim/traces_per_session


## 2. Clean the working population — drop L1, add label columns

Decisions taken (recorded here for the record):
- **L1 → drop.** All 15 are `cell_type=23P` but the geometric segmenter placed
  them above L2/3. WORKFLOW §2.1 doesn't include L1; we follow the workflow.
- **Phase-1 label = `layer_label` (geometric layer).** Trusts the structural
  segmenter directly.
- **Phase-3 label = `celltype_label` (AIBS cell-type).** When the two disagree
  (~700 neurons), within-layer subtype tasks read `celltype_label`, not the
  geometric layer.

In [2]:
units = pd.read_parquet(PROCESSED_TABLES_DIR / 'units_v1_exc_best.parquet')
print('rows in working pop (pre-clean):', len(units))

# Drop L1.
before = len(units)
units = units[units['layer'] != 'L1'].copy()
print(f'dropped L1: {before - len(units):,} -> {len(units):,} rows remaining')

# Define labels explicitly, decoupled from the raw `layer` / `cell_type` columns
# so downstream code never has to second-guess which to use.
units['layer_label']    = units['layer']      # Phase-1 target
units['celltype_label'] = units['cell_type']  # Phase-3 target

# Build a stable session_key string (the H5 keys it on this).
units['session_key'] = units['session'].astype(int).astype(str) + '_' + units['scan_idx'].astype(int).astype(str)
units['unit_id'] = units['unit_id'].astype(int)

# Sanity: each nucleus should appear exactly once (best-only).
assert units['nucleus_id'].is_unique, 'nucleus_id is not unique in best_only — bug?'

# Save cleaned working population. This is the canonical neuron-level table
# every later notebook reads.
out = PROCESSED_TABLES_DIR / 'units_working.parquet'
units.to_parquet(out)
print('saved:', out)
print()
print('layer_label distribution:')
print(units['layer_label'].value_counts().sort_index())
print()
print('celltype_label distribution:')
print(units['celltype_label'].value_counts())
print()
print('active session_keys:', sorted(units['session_key'].unique()))

rows in working pop (pre-clean): 8910
dropped L1: 15 -> 8,895 rows remaining
saved: data/processed/tables/units_working.parquet

layer_label distribution:
layer_label
L2/3    4247
L4      2670
L5      1615
L6       363
Name: count, dtype: int64

celltype_label distribution:
celltype_label
23P      4198
4P       2845
5P-IT    1169
5P-ET     320
6P-IT     216
6P-CT     121
5P-NP      26
Name: count, dtype: int64

active session_keys: ['4_7', '5_6', '5_7', '6_2', '6_4', '6_6', '6_7', '7_3', '7_5', '8_5', '9_3', '9_4', '9_6']


## 3. Build a `condition_hash` → stim type lookup, once

The H5's per-session `meta/condition_hashes` gives the hash of each trial. To
tag each trial with its stimulus family (Clip / Monet2 / Trippy) without
calling the slow `get_video_type` once per trial, we build a hash → family
dict from `f['types/<family>']` once.

In [3]:
def build_hash_to_family_map(reader):
    h2f: dict[str, str] = {}
    for fam in STIMULUS_FAMILIES:
        try:
            for h in reader.get_hashes_by_type(fam):
                # `get_hashes_by_type` may yield bytes or str — normalise.
                if isinstance(h, bytes):
                    h = h.decode()
                h2f[h] = fam
        except Exception as e:
            print(f'WARN: get_hashes_by_type({fam}) failed: {e}')
    return h2f

reader = mic.MicronsFunctionalReader(
    datadir=str(DATA_DIR),
    path=str(FUNCTIONAL_H5),
)
hash_to_family = build_hash_to_family_map(reader)
print(f'hash_to_family covers {len(hash_to_family):,} hashes; sample 3:')
for k, v in list(hash_to_family.items())[:3]:
    print(f'  {k!r} -> {v}')

hash_to_family covers 2,411 hashes; sample 3:
  '/0MJJl6hCzFo3gxCJSmv' -> Clip
  '/3cVqNzav4cRdcLPJXWr' -> Clip
  '/5KYggaBDxG7uxMGf589' -> Clip


## 4. Stream the H5 — one session at a time, checkpointed

For each active `session_key` we:
1. Read `meta/unit_ids` once and build a row-index map for our working-pop
   neurons in that session.
2. Read `meta/condition_hashes` to tag each trial.
3. Iterate over trials, reading `responses` (sliced to working-pop rows),
   `treadmill`, `pupil`, and `stim_times`.
4. Append rows to two builders: a per-neuron-per-trial response builder, and
   a per-trial behaviour builder.
5. Write two per-session checkpoint parquets to `data/interim/traces_per_session/`
   (`responses_<sk>.parquet`, `trials_<sk>.parquet`).

If the cell is interrupted, simply re-run — sessions whose checkpoints already
exist are skipped.

The two-table split avoids storing pupil/treadmill 8,910 times per trial
(they are shared across all neurons in a trial).

In [4]:
def collect_session(h5_file, session_key, working_units_in_session, hash_to_family,
                    out_responses_path, out_trials_path):
    """Process one session_key. Writes two parquets and returns row counts."""
    grp = h5_file[f'sessions/{session_key}']
    h5_unit_ids = np.asarray(grp['meta/unit_ids'])
    cond_hashes_raw = np.asarray(grp['meta/condition_hashes'])
    n_trials = len(cond_hashes_raw)
    # Map our working-pop unit_ids -> H5 row index.
    unit_id_to_row: dict[int, int] = {
        int(uid): int(idx) for idx, uid in enumerate(h5_unit_ids)
    }
    pop_unit_ids = working_units_in_session['unit_id'].astype(int).tolist()
    pop_nucleus_ids = working_units_in_session['nucleus_id'].astype(int).tolist()

    # Drop any working-pop unit not present in this session's H5 unit list
    # (shouldn't happen with best_only, but guard).
    keep_idx = [i for i, uid in enumerate(pop_unit_ids) if uid in unit_id_to_row]
    if len(keep_idx) != len(pop_unit_ids):
        missing = len(pop_unit_ids) - len(keep_idx)
        print(f'  WARN: {missing} working-pop unit_ids not in H5 unit list for {session_key}')
    pop_unit_ids = [pop_unit_ids[i] for i in keep_idx]
    pop_nucleus_ids = [pop_nucleus_ids[i] for i in keep_idx]
    pop_h5_rows = np.array([unit_id_to_row[uid] for uid in pop_unit_ids], dtype=np.int64)

    n_pop = len(pop_unit_ids)
    print(f'  session {session_key}: {n_pop} working-pop neurons, {n_trials} trials')

    # Builders.
    resp_rows = []  # per (nucleus_id, trial_idx)
    trial_rows = []  # per (session_key, trial_idx)

    # Iterate trials.
    trials_grp = grp['trials']
    for ti in range(n_trials):
        t = trials_grp[str(ti)]
        responses = np.asarray(t['responses'], dtype=np.float32)  # (n_units, n_frames)
        treadmill = np.asarray(t['treadmill'], dtype=np.float32)  # (n_frames, 1) or (n_frames,)
        pupil     = np.asarray(t['pupil'],     dtype=np.float32)  # (4, n_frames)
        stim_times= np.asarray(t['stim_times'],dtype=np.float32)  # (n_frames,)

        ch_raw = cond_hashes_raw[ti]
        ch = ch_raw.decode() if isinstance(ch_raw, (bytes, bytearray)) else str(ch_raw)
        stim_type = hash_to_family.get(ch, '<unknown>')
        n_frames = int(responses.shape[1])

        # Slice responses to working-pop rows.
        sub_resp = responses[pop_h5_rows, :]  # (n_pop, n_frames)

        # Append per-neuron rows. We store the trace as a list[float32].
        for j in range(n_pop):
            resp_rows.append({
                'nucleus_id'   : pop_nucleus_ids[j],
                'session_key'  : session_key,
                'trial_idx'    : ti,
                'condition_hash': ch,
                'stim_type'    : stim_type,
                'n_frames'     : n_frames,
                'response'     : sub_resp[j].tolist(),
            })

        # Trial-level shared meta + behaviour.
        # Normalise treadmill to 1D.
        tread_1d = treadmill.reshape(-1).astype(np.float32)
        trial_rows.append({
            'session_key'   : session_key,
            'trial_idx'     : ti,
            'condition_hash': ch,
            'stim_type'     : stim_type,
            'n_frames'      : n_frames,
            'treadmill'     : tread_1d.tolist(),
            'pupil_pos_x'   : pupil[0].astype(np.float32).tolist(),
            'pupil_pos_y'   : pupil[1].astype(np.float32).tolist(),
            'pupil_dilation': pupil[2].astype(np.float32).tolist(),
            'pupil_aux'     : pupil[3].astype(np.float32).tolist(),
            'stim_times'    : stim_times.astype(np.float32).tolist(),
        })

    df_resp  = pd.DataFrame(resp_rows)
    df_trial = pd.DataFrame(trial_rows)
    df_resp.to_parquet(out_responses_path, compression='zstd')
    df_trial.to_parquet(out_trials_path, compression='zstd')
    return len(df_resp), len(df_trial)

In [5]:
# Group working-pop neurons by their session_key.
by_sess = {sk: g for sk, g in units.groupby('session_key')}
session_keys = sorted(by_sess.keys())
print('active session_keys:', session_keys)
print()

summary_rows = []
t0 = time.time()
with h5py.File(FUNCTIONAL_H5, 'r') as f:
    for sk in session_keys:
        out_resp  = INTERIM_TRACES_DIR / f'responses_{sk}.parquet'
        out_trial = INTERIM_TRACES_DIR / f'trials_{sk}.parquet'
        if out_resp.exists() and out_trial.exists():
            print(f'  session {sk}: SKIP (checkpoints already exist)')
            n_r = pq.read_metadata(out_resp).num_rows
            n_t = pq.read_metadata(out_trial).num_rows
            summary_rows.append({'session_key': sk, 'n_response_rows': n_r, 'n_trial_rows': n_t, 'skipped': True})
            continue
        n_r, n_t = collect_session(f, sk, by_sess[sk], hash_to_family, out_resp, out_trial)
        summary_rows.append({'session_key': sk, 'n_response_rows': n_r, 'n_trial_rows': n_t, 'skipped': False})
        print(f'  -> wrote {n_r:,} response rows, {n_t:,} trial rows  '
              f'(elapsed {time.time() - t0:.1f}s)')
        gc.collect()

print()
print(f'total elapsed: {time.time() - t0:.1f}s')
pd.DataFrame(summary_rows)

active session_keys: ['4_7', '5_6', '5_7', '6_2', '6_4', '6_6', '6_7', '7_3', '7_5', '8_5', '9_3', '9_4', '9_6']

  session 4_7: 715 working-pop neurons, 464 trials
  -> wrote 331,760 response rows, 464 trial rows  (elapsed 8.5s)
  session 5_6: 686 working-pop neurons, 464 trials
  -> wrote 318,304 response rows, 464 trial rows  (elapsed 17.3s)
  session 5_7: 701 working-pop neurons, 464 trials
  -> wrote 325,264 response rows, 464 trial rows  (elapsed 25.4s)
  session 6_2: 774 working-pop neurons, 464 trials
  -> wrote 359,136 response rows, 464 trial rows  (elapsed 33.9s)
  session 6_4: 916 working-pop neurons, 464 trials
  -> wrote 425,024 response rows, 464 trial rows  (elapsed 43.4s)
  session 6_6: 697 working-pop neurons, 464 trials
  -> wrote 323,408 response rows, 464 trial rows  (elapsed 51.3s)
  session 6_7: 874 working-pop neurons, 464 trials
  -> wrote 405,536 response rows, 464 trial rows  (elapsed 59.9s)
  session 7_3: 552 working-pop neurons, 464 trials
  -> wrote 256,12

,session_key,n_response_rows,n_trial_rows,skipped
0,4_7,331760,464,False
1,5_6,318304,464,False
2,5_7,325264,464,False
3,6_2,359136,464,False
4,6_4,425024,464,False
5,6_6,323408,464,False
6,6_7,405536,464,False
7,7_3,256128,464,False
8,7_5,127136,464,False
9,8_5,361920,464,False


## 5. Combine per-session checkpoints into the canonical caches

Concatenate the per-session response and trial parquets into
`traces.parquet` and `trials_meta.parquet`. These are the canonical inputs
that every later feature notebook reads.

In [6]:
resp_paths  = sorted(INTERIM_TRACES_DIR.glob('responses_*.parquet'))
trial_paths = sorted(INTERIM_TRACES_DIR.glob('trials_*.parquet'))
print(f'concatenating {len(resp_paths)} response parquets, {len(trial_paths)} trial parquets …')

df_traces = pd.concat([pd.read_parquet(p) for p in resp_paths], ignore_index=True)
df_trials = pd.concat([pd.read_parquet(p) for p in trial_paths], ignore_index=True)

print()
print('traces.parquet     :', df_traces.shape, '| neurons:',
      df_traces['nucleus_id'].nunique(),
      '| trials per neuron (median):',
      int(df_traces.groupby('nucleus_id').size().median()))
print('trials_meta.parquet:', df_trials.shape, '| trials:',
      df_trials.groupby(['session_key', 'trial_idx']).ngroups)

out_traces = PROCESSED_TABLES_DIR / 'traces.parquet'
out_trials = PROCESSED_TABLES_DIR / 'trials_meta.parquet'
df_traces.to_parquet(out_traces, compression='zstd')
df_trials.to_parquet(out_trials, compression='zstd')
print()
print('saved:', out_traces, '|', round(out_traces.stat().st_size / 1e6, 1), 'MB')
print('saved:', out_trials, '|', round(out_trials.stat().st_size / 1e6, 1), 'MB')

concatenating 13 response parquets, 13 trial parquets …

traces.parquet     : (4127280, 7) | neurons: 8895 | trials per neuron (median): 464
trials_meta.parquet: (6032, 11) | trials: 6032

saved: data/processed/tables/traces.parquet | 1658.4 MB
saved: data/processed/tables/trials_meta.parquet | 10.1 MB


## 6. Within-hash trial-averaged traces (A1 substrate)

WORKFLOW §3.3 makes a strong distinction:

- **Within-hash averaging is signal estimation** — the canonical PSTH-style
  estimator of the stimulus-locked response per `(neuron, hash)`.
- **Across-hash averaging is destructive** — forbidden as a feature step.

We compute the within-hash average here, alongside the trial count `n_trials`
(needed for any reliability statistic: weighted means, oracle correlation,
Fano factor, etc., which are computed in a later notebook).

**No across-hash averaging is performed anywhere in this notebook.**

In [7]:
def avg_traces_within_hash(df_traces: pd.DataFrame) -> pd.DataFrame:
    """For each (nucleus_id, condition_hash), average the per-trial response
    traces. Trials of the same hash always have the same n_frames inside one
    stim family, so a simple stack-and-mean works."""
    out = []
    grouped = df_traces.groupby(['nucleus_id', 'condition_hash'], sort=False)
    n = len(grouped)
    for i, ((nid, ch), grp) in enumerate(grouped):
        if i % 50_000 == 0 and i > 0:
            print(f'  averaged {i:,}/{n:,} (neuron, hash) pairs …')
        # All trials of one hash share a frame count.
        traces = np.stack([np.asarray(r, dtype=np.float32) for r in grp['response'].values])
        avg = traces.mean(axis=0)
        out.append({
            'nucleus_id'   : nid,
            'condition_hash': ch,
            'session_key'  : grp['session_key'].iloc[0],
            'stim_type'    : grp['stim_type'].iloc[0],
            'n_frames'     : int(traces.shape[1]),
            'n_trials'     : int(traces.shape[0]),
            'response_avg' : avg.tolist(),
        })
    return pd.DataFrame(out)

print('computing within-hash averages …')
t0 = time.time()
df_avg = avg_traces_within_hash(df_traces)
print(f'  done in {time.time() - t0:.1f}s; shape = {df_avg.shape}')

out_avg = PROCESSED_TABLES_DIR / 'traces_avg.parquet'
df_avg.to_parquet(out_avg, compression='zstd')
print('saved:', out_avg, '|', round(out_avg.stat().st_size / 1e6, 1), 'MB')

# Quick sanity numbers.
print()
print('rows per stim_type (avg table):')
print(df_avg['stim_type'].value_counts())
print()
print('hashes per neuron (avg table) — distribution:')
hpn = df_avg.groupby('nucleus_id').size()
print(hpn.describe().round(1))
print()
print('trials per (neuron, hash) — distribution:')
print(df_avg['n_trials'].describe().round(1))

computing within-hash averages …
  averaged 50,000/2,490,600 (neuron, hash) pairs …
  averaged 100,000/2,490,600 (neuron, hash) pairs …
  averaged 150,000/2,490,600 (neuron, hash) pairs …
  averaged 200,000/2,490,600 (neuron, hash) pairs …
  averaged 250,000/2,490,600 (neuron, hash) pairs …
  averaged 300,000/2,490,600 (neuron, hash) pairs …
  averaged 350,000/2,490,600 (neuron, hash) pairs …
  averaged 400,000/2,490,600 (neuron, hash) pairs …
  averaged 450,000/2,490,600 (neuron, hash) pairs …
  averaged 500,000/2,490,600 (neuron, hash) pairs …
  averaged 550,000/2,490,600 (neuron, hash) pairs …
  averaged 600,000/2,490,600 (neuron, hash) pairs …
  averaged 650,000/2,490,600 (neuron, hash) pairs …
  averaged 700,000/2,490,600 (neuron, hash) pairs …
  averaged 750,000/2,490,600 (neuron, hash) pairs …
  averaged 800,000/2,490,600 (neuron, hash) pairs …
  averaged 850,000/2,490,600 (neuron, hash) pairs …
  averaged 900,000/2,490,600 (neuron, hash) pairs …
  averaged 950,000/2,490,600 (ne

## 7. CV splits — `StratifiedGroupKFold` + `LeaveOneSessionOut`

WORKFLOW §6.1 / §6.1bis demand:
- Splits grouped by `nucleus_id` so the same neuron never appears on both
  sides of a fold (the row-level → neuron-level mismatch).
- Stratified by `layer_label` to keep class balance comparable across folds.
- A separate `LeaveOneSessionOut` protocol as a sanity check, because scan
  composition correlates with the layer label.

Both protocols are saved as **per-neuron** assignments. Modelling notebooks
translate fold ↔ row by joining on `nucleus_id`.

In [8]:
from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 5

# Build a neuron-level frame keyed by nucleus_id, with the columns the splitter needs.
neurons = (units[['nucleus_id', 'layer_label', 'session_key']]
           .drop_duplicates(subset='nucleus_id')
           .reset_index(drop=True))
print('neurons table:', neurons.shape)

skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
fold_assignments = np.full(len(neurons), -1, dtype=np.int8)
for fold, (_train, test_idx) in enumerate(
        skf.split(np.zeros(len(neurons)), neurons['layer_label'], groups=neurons['nucleus_id'])):
    fold_assignments[test_idx] = fold

neurons['gkf_fold'] = fold_assignments
assert (neurons['gkf_fold'] >= 0).all(), 'every neuron should be assigned a test fold'

# LOSO assignment: the neuron's session_key is its leave-out fold.
neurons['loso_test_scan'] = neurons['session_key']

out_splits = PROCESSED_SPLITS_DIR / 'cv_assignments.parquet'
neurons.to_parquet(out_splits)
print('saved:', out_splits)

print()
print('GKF fold sizes:')
print(neurons['gkf_fold'].value_counts().sort_index())
print()
print('GKF class balance per fold:')
print(pd.crosstab(neurons['gkf_fold'], neurons['layer_label']))
print()
print('LOSO scan sizes:')
print(neurons['loso_test_scan'].value_counts().sort_index())

neurons table: (8895, 3)
saved: data/processed/splits/cv_assignments.parquet

GKF fold sizes:
gkf_fold
0    1779
1    1779
2    1779
3    1779
4    1779
Name: count, dtype: int64

GKF class balance per fold:
layer_label  L2/3   L4   L5  L6
gkf_fold                       
0             834  549  325  71
1             851  528  336  64
2             856  533  314  76
3             851  535  314  79
4             855  525  326  73

LOSO scan sizes:
loso_test_scan
4_7    715
5_6    686
5_7    701
6_2    774
6_4    916
6_6    697
6_7    874
7_3    552
7_5    274
8_5    780
9_3    818
9_4    680
9_6    428
Name: count, dtype: int64


## 8. Conclusions

Three canonical caches in place:

- `units_working.parquet` — cleaned working population (L1 dropped) with
  `layer_label` (Phase 1) and `celltype_label` (Phase 3) columns.
- `traces.parquet` + `trials_meta.parquet` — calcium traces per
  `(nucleus_id, trial_idx)` and shared trial-level behaviour per
  `(session_key, trial_idx)`. Together they are the substrate for every later
  feature tier.
- `traces_avg.parquet` — within-hash trial-averaged response per
  `(nucleus_id, condition_hash)`. The PSTH-style signal estimator the
  workflow names A1; per-trial reliability statistics will be computed from
  the original `traces.parquet` in a later notebook (Tier B).
- `cv_assignments.parquet` — per-neuron `gkf_fold` and `loso_test_scan`.

**Decisions that flowed into this notebook (kept here for traceability).**
1. Drop L1 (15 neurons, all `cell_type=23P`).
2. Phase-1 label = geometric `layer`; Phase-3 label = AIBS `cell_type`.
3. Cache raw traces (Choice A) so future tiers don't re-read the H5.

**What `03_tier_a.ipynb` will do next.**
- Read `traces.parquet` and `trials_meta.parquet`.
- Build Tier A0 (per-trial scalar features: amplitude, peak, integral,
  latency, rise/decay, early/late AUC, adaptation) and Tier A1 (the same
  scalars on the within-hash averaged trace from `traces_avg.parquet`).
- Save one parquet per sub-block (`A0_amp`, `A0_shape`, `A1_amp`, `A1_shape`)
  under `data/processed/features/` so the Stage-1 ablation in Phase 1 can
  combine and ablate them cleanly.